# IMPORTS

In [1]:
!pip install timm
!pip install ttach
!pip install imagededup


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os; import shutil; import subprocess; import torch; import torch.nn as nn
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt; import seaborn as sns
from datetime import datetime, timedelta, timezone; from pathlib import Path
from torch.utils.data import DataLoader, random_split; from torchvision import transforms, models, datasets
from torchvision.models import EfficientNet_V2_S_Weights, ConvNeXt_Tiny_Weights
from tqdm.auto import tqdm; from PIL import Image; from IPython import get_ipython
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                             precision_recall_curve, average_precision_score, accuracy_score)
from sklearn.preprocessing import label_binarize; from scipy.signal import savgol_filter
import kagglehub; import gc; import sys; import inspect; import pprint; import stat
from matplotlib.gridspec import GridSpec; from matplotlib.collections import LineCollection
from matplotlib.ticker import MaxNLocator; from imagededup.methods import PHash
import ttach as tta
import timm
if 'google.colab' in sys.modules:
    print("Running in Google Colab")
    is_colab = True
    from google.colab import _message
    from google.colab import userdata
else:
    print("Not running in Google Colab")
    is_colab = False
    torch.hub.set_dir('weights')
from imagededup.methods import PHash

Not running in Google Colab


# ДАННЫЕ

In [3]:
if is_colab:
    token = userdata.get('GITHUB_DLTESTWORK_TOKEN')
else:
    token = os.environ.get('GITHUB_DLTESTWORK_TOKEN')
    os.environ["KAGGLEHUB_CACHE"] = "datasets"
path_filtered = kagglehub.dataset_download("kvdep1/dumplings")
path_sub = kagglehub.competition_download("yummi-classification-fu25")

100%|██████████| 395M/395M [00:13<00:00, 30.3MB/s] 

Extracting files...


100%|██████████| 5.96M/5.96M [00:01<00:00, 3.95MB/s]

Extracting files...


# LOGGING

In [15]:
class GitLogger:
    def __init__(self, repo, tok, folder="dumplings"):
        self.raw_token = str(tok).strip()
        self.u_mail, self.u_name = os.environ.get('GIT_EMAIL', 'bot@exp.com'), os.environ.get('GIT_NAME', 'Bot')
        self.base = folder; tok = str(tok).strip(); repo = repo.strip().replace("https://","")
        self.url = f"https://{tok}@{repo}"; self.local = os.path.join(os.getcwd(), repo.split("/")[-1].replace(".git",""))
        self._init_repo(); self.exp_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        self.path = os.path.join(self.local, self.base, self.exp_name); os.makedirs(self.path, exist_ok=True)

    def _init_repo(self):
        if os.path.exists(self.local):
            def rm_err(f, p, e): os.chmod(p, stat.S_IWRITE); f(p)
            shutil.rmtree(self.local, onerror=rm_err)
        subprocess.run(["git","clone","--filter=blob:none","--no-checkout","--depth=1",self.url,self.local], check=True, capture_output=True)
        subprocess.run(["git","-C",self.local,"config","user.email",self.u_mail], check=False)
        subprocess.run(["git","-C",self.local,"config","user.name",self.u_name], check=False)
        subprocess.run(["git","-C",self.local,"reset","--mixed","HEAD"], check=True, capture_output=True)

    def start(self, metric=None):
        if metric:
            new = f"{metric:.4f}_{self.exp_name}"; n_path = os.path.join(self.local, self.base, new)
            os.rename(self.path, n_path); self.path = n_path; self.exp_name = new
        with open(os.path.join(self.path, "info.txt"), "w") as f: f.write(f"Start: {metric}")

    def push(self, msg="Log"):
        try:
            subprocess.run(["git","-C",self.local,"add",f"{self.base}/{self.exp_name}"], check=True)
            subprocess.run(["git","-C",self.local,"commit","-m",f"{msg} {self.exp_name}"], check=True)
            subprocess.run(["git","-C",self.local,"push"], check=True); print("Pushed.")
        except Exception as e: print(f"Push fail: {e}")

    def save_code(self):
        if not self.path: return
        target = os.path.join(self.path, "run_code.py")

        try:
            full_text = ""

            # CASE 1: Google Colab
            if 'google.colab' in sys.modules:
                from google.colab import _message # Import inside function to be safe
                nb = _message.blocking_request('get_ipynb')
                code_content = []
                if nb and 'ipynb' in nb and 'cells' in nb['ipynb']:
                    for cell in nb['ipynb']['cells']:
                        if cell['cell_type'] == 'code':
                            source = "".join(cell['source'])
                            code_content.append(source)
                full_text = "\n\n# %% [CELL] \n\n".join(code_content)

            # CASE 2: Local Script (.py)
            else:
                # If running as a .py script, read the file itself
                try:
                    with open(sys.argv[0], 'r') as f:
                        full_text = f.read()
                except:
                    full_text = "# Could not read source file"

            # Redact Token
            if self.raw_token and len(self.raw_token) > 3:
                full_text = full_text.replace(self.raw_token, "REDACTED_TOKEN")

            with open(target, 'w', encoding='utf-8') as f:
                f.write(full_text)

            print(f"[INFO] Code saved to {target}")

        except Exception as e:
            print(f"[ERROR] Failed to save code: {e}")
    def save_csv(self, df, n): df.to_csv(os.path.join(self.path, n), index=False)
    def save_fig(self, fig, n): fig.savefig(os.path.join(self.path, n))

# DATA + MODEL

In [6]:
def clean_duplicates(img_dir, thresh=10):
    print(f"Deduping {img_dir}..."); phasher = PHash()
    encs = phasher.encode_images(image_dir=img_dir, recursive=True)
    dups = phasher.find_duplicates(encoding_map=encs, max_distance_threshold=thresh)
    rm_files = set([d for sublist in dups.values() for d in sublist])
    print(f"Removing {len(rm_files)}..."); [os.remove(Path(img_dir)/f) for f in rm_files if (Path(img_dir)/f).exists()]

if is_colab:
    def to_rgb(img): return img.convert('RGB')
    def get_dataloaders(CFG):
        tf_tr = transforms.Compose([
            transforms.Lambda(to_rgb),
            transforms.RandomResizedCrop(CFG['img_size'], scale=(0.5,1.0), ratio=(0.75,1.33)),
            transforms.RandomHorizontalFlip(p=0.5), transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(45),
            transforms.RandomPerspective(distortion_scale=0.4, p=0.4),
            transforms.RandomAffine(0, translate=(0.1,0.1), shear=15),
            transforms.RandomApply([transforms.GaussianBlur((5,9), sigma=(0.1,5))], p=0.3),
            transforms.ColorJitter(0.3,0.3,0.3,0.1), transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.3, scale=(0.02,0.2))
        ])
        tf_val = transforms.Compose([
            transforms.Lambda(to_rgb), transforms.Resize((CFG['img_size'], CFG['img_size'])),
            transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        fd_tr = datasets.ImageFolder(CFG['data_dir'], transform=tf_tr)
        fd_val = datasets.ImageFolder(CFG['data_dir'], transform=tf_val)
        gen = torch.Generator().manual_seed(42)
        tr_ds, _ = random_split(fd_tr, [int(0.8*len(fd_tr)), len(fd_tr)-int(0.8*len(fd_tr))], generator=gen)
        _, val_ds = random_split(fd_val, [int(0.8*len(fd_tr)), len(fd_tr)-int(0.8*len(fd_tr))], generator=gen)
        return (DataLoader(tr_ds, CFG['batch_size'], shuffle=True, num_workers=2, pin_memory=True),
                DataLoader(val_ds, CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True), fd_tr.classes)
else:
    from dataloaders import get_dataloaders

def build_model(arch, n_cls, device):
    print(f"Building {arch}...")
    if arch == 'effnet':
        # EfficientNet V2 Small (Хорошая база)
        m = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_cls)

    elif arch == 'convnext':
        # ConvNeXt Tiny (Понимает текстуры)
        m = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_cls)

    elif arch == 'swin':
        # Swin Transformer V2 Small (Иерархический трансформер, лучше видит форму)
        m = models.swin_v2_s(weights='DEFAULT')
        m.head = nn.Linear(m.head.in_features, n_cls)

    elif arch == 'maxvit':
        # MaxViT Tiny (Гибрид CNN + Attention, SOTA на ImageNet для малых размеров)
        m = models.maxvit_t(weights='DEFAULT')
        # У MaxViT классификатор это Sequential, последний слой - Linear
        m.classifier[5] = nn.Linear(m.classifier[5].in_features, n_cls)

    elif arch == 'eva02':
        # EVA-02 Small (MIM pretraining - очень мощный геометрический инвариант)
        # Требует img_size=336 для лучшего качества
        m = timm.create_model('eva02_small_patch14_336.mim_in22k_ft_in1k', pretrained=True, num_classes=n_cls)

    elif arch == 'dinov2':
        # DINOv2 (ViT Small) - обучалась без учителя.
        # Используем регистры (reg) для лучшего внимания
        m = timm.create_model('vit_small_patch14_reg4_dinov2.lvd142m', pretrained=True, num_classes=n_cls)

    return m.to(device)

# TRAIN

In [7]:
def train(CFG):
    tr_dl, val_dl, cls = get_dataloaders(CFG)
    model = build_model(CFG['arch'], len(cls), CFG['device'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], steps_per_epoch=len(tr_dl), epochs=CFG['epochs'])
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {'tr_loss':[], 'val_acc':[]}; best = 0.0

    print(f"Training {CFG['arch']}...")
    for ep in range(CFG['epochs']):
        model.train(); ep_loss = 0
        for x, y in tqdm(tr_dl, desc=f"Ep {ep+1}", leave=False):
            x,y = x.to(CFG['device']), y.to(CFG['device']); opt.zero_grad()
            loss = crit(model(x), y); loss.backward(); opt.step(); sched.step(); ep_loss += loss.item()

        model.eval(); corr = 0; tot = 0
        with torch.no_grad():
            for x, y in val_dl:
                x,y = x.to(CFG['device']), y.to(CFG['device']); _, p = torch.max(model(x), 1)
                corr += (p==y).sum().item(); tot += y.size(0)

        acc = corr/tot; hist['tr_loss'].append(ep_loss/len(tr_dl)); hist['val_acc'].append(acc)
        print(f"Ep {ep+1} | Loss: {hist['tr_loss'][-1]:.4f} | Acc: {acc:.4f}")
        if acc > best: best = acc; torch.save(model.state_dict(), f"best_{CFG['arch']}.pth")

    return model, val_dl, cls, hist, best

# VIS

In [8]:
def viz_ensemble(models_list, val_dl, classes, logger, device):
    print("Visualizing Ensemble..."); [m.eval() for m in models_list]
    y_true, y_prob = [], []

    with torch.no_grad():
        for x, y in val_dl:
            x = x.to(device); preds = torch.zeros(x.size(0), len(classes)).to(device)
            for m in models_list: preds += torch.softmax(m(x), dim=1)
            y_true.extend(y.cpu().numpy()); y_prob.extend((preds/len(models_list)).cpu().numpy())

    y_true, y_prob = np.array(y_true), np.array(y_prob)

    plt.rcParams.update({'font.family':'sans-serif','font.weight':'bold','figure.facecolor':'#0a0a0a','text.color':'white','axes.labelcolor':'white','axes.edgecolor':'#333','axes.facecolor':'#1a1a1a','xtick.color':'white','ytick.color':'white'})

    fig = plt.figure(figsize=(20, 10)); gs = GridSpec(2, 3)

    # 1. Dist
    ax1 = fig.add_subplot(gs[0, 0])
    [sns.kdeplot(y_prob[:,i], fill=True, label=classes[i], ax=ax1) for i in range(len(classes))]
    ax1.set_title("Conf. Dist"); ax1.legend(facecolor='#1a1a1a', labelcolor='white')

    # 2. CM
    ax2 = fig.add_subplot(gs[0, 1])
    sns.heatmap(confusion_matrix(y_true, np.argmax(y_prob, 1)), annot=True, fmt='d', cmap='magma', ax=ax2, cbar=False, annot_kws={'color':'white'})
    ax2.set_title("Ensemble CM")

    # 3. Entropy
    ax3 = fig.add_subplot(gs[1, :])
    ent = -np.sum(y_prob * np.log(y_prob+1e-10), 1)
    ax3.scatter(range(len(ent)), ent, c=ent, cmap='coolwarm', s=10); ax3.set_title("Uncertainty (Entropy)")

    logger.save_fig(fig, "ensemble_report.png"); plt.close(fig)


def run_full_evaluation(model, loader, logger, history, classes, config):
    print("Running Full Evaluation..."); model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(config['device']); out = model(x); prob = torch.softmax(out, dim=1)
            y_true.extend(y.cpu().numpy()); y_pred.extend(torch.max(prob, 1)[1].cpu().numpy()); y_prob.extend(prob.cpu().numpy())

    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob); n_classes = len(classes)

    # Styles
    MAIN_RED = '#C62828'; CLS_COLORS = ['#C62828', '#1565C0', '#F9A825', '#2E7D32']
    plt.rcParams.update({'font.family':'sans-serif', 'font.weight':'bold', 'font.size':14, 'axes.titlesize':26, 'axes.labelsize':18,
                         'xtick.labelsize':14, 'ytick.labelsize':14, 'legend.fontsize':14, 'figure.facecolor':'white',
                         'axes.facecolor':'white', 'text.color':'black', 'axes.labelcolor':'black', 'axes.edgecolor':'#333',
                         'axes.grid':True, 'grid.alpha':0.2, 'grid.color':'black'})

    fig = plt.figure(figsize=(24, 20), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1], width_ratios=[0.9, 1.2, 1])
    fig.suptitle(f"Evaluation Report | {(datetime.now(timezone.utc) + timedelta(hours=3)).strftime('%Y-%m-%d %H:%M')}", fontsize=48, y=1.3, fontweight='bold')

    def plot_gradient_line(ax, x, y, title, ylabel=None, rising_is_good=True, is_loss=False):
        if not is_loss: ax.set_ylim(0, 1.02)
        ax.set_xlim(0, max(x)+1 if len(x)>0 else 2); ax.set_xticks(np.arange(1, max(x)+1 if len(x)>0 else 2))

        if len(x) > 1:
            pts = np.array([x, y]).T.reshape(-1, 1, 2)
            segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
            cols = [MAIN_RED if (d >= 0 if rising_is_good else d <= 0) else 'black' for d in np.diff(y)]
            ax.add_collection(LineCollection(segs, colors=cols, linewidths=5))
            best_idx = np.argmax(y) if not is_loss else np.argmin(y)
        else:
            best_idx = 0

        tx, ty = (x[best_idx], y[best_idx]) if len(x) > 0 else (0,0)
        ax.scatter(tx, ty, color='black', s=200, zorder=5)
        ax.annotate(f"{ty:.4f}", (tx, ty), textcoords="offset points", xytext=(0, 15), ha='center', color='black', fontweight='bold', fontsize=20)
        ax.set_title(title, pad=20, fontsize=36, fontweight='bold')
        if ylabel: ax.set_ylabel(ylabel)

    # A. Accuracy
    ax_acc = fig.add_subplot(gs[0, :2])
    plot_gradient_line(ax_acc, np.arange(1, len(history['val_acc']) + 1), np.array(history['val_acc']), 'Accuracy', rising_is_good=True)

    # B. ROC
    ax_roc = fig.add_subplot(gs[0, 2]); y_bin = label_binarize(y_true, classes=range(n_classes))
    for i in range(n_classes):
        if np.sum(y_bin[:, i]) > 0:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            ax_roc.plot(fpr, tpr, lw=4, color=CLS_COLORS[i % 4], label=f'{classes[i]} ({auc(fpr, tpr):.2f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.4); ax_roc.set_title('ROC Curves', fontsize=36, fontweight='bold')
    ax_roc.legend(loc='lower right', frameon=False, fontsize=25)

    # C. Confusion Matrix
    ax_cm = fig.add_subplot(gs[1, 0]); cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=sns.blend_palette(['white', '#dddddd', 'black', MAIN_RED], as_cmap=True),
                ax=ax_cm, cbar=False, xticklabels=classes, yticklabels=classes, square=True, annot_kws={"size": 24, "weight": "bold"})
    [t.set_color('white' if int(t.get_text()) > cm.max()/2 else 'black') for t in ax_cm.texts]
    ax_cm.set_title('Confusion Matrix', pad=20, fontsize=36, fontweight='bold'); ax_cm.set_ylabel('Actual'); ax_cm.set_xlabel('Predicted')

    # D. Metrics Heatmap
    ax_met = fig.add_subplot(gs[1:, 1])
    rep = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    df_met = pd.DataFrame([[rep[c]['precision'], rep[c]['recall'], rep[c]['f1-score']] for c in classes], columns=['Prec', 'Recall', 'F1'])
    sns.heatmap(df_met, annot=True, fmt='.2f', cmap=sns.dark_palette(MAIN_RED, as_cmap=True, input='rgb'), ax=ax_met, cbar=False, annot_kws={"size": 36, "weight": "bold", "color": "white"})
    ax_met.xaxis.tick_top(); ax_met.set_xticklabels(df_met.columns, fontsize=36, fontweight='bold'); ax_met.set_yticks([])
    [ax_met.text(-0.1, i + 0.5, c.upper(), fontsize=36, fontweight='bold', rotation=90, va='center', ha='right', transform=ax_met.transData) for i, c in enumerate(classes)]

    # E. Confidence
    ax_hist = fig.add_subplot(gs[1, 2]); max_p = np.max(y_prob, axis=1); msk = (y_pred == y_true); bins = np.linspace(0.3, 1, 15)
    ax_hist.hist(max_p[~msk], bins=bins, alpha=0.6, color='black', label='Wrong', edgecolor='white')
    ax_hist.hist(max_p[msk], bins=bins, alpha=0.6, color=MAIN_RED, label='Correct', edgecolor='white')
    ax_hist.set_title('Confidence Distribution', pad=15, fontsize=36, fontweight='bold'); ax_hist.legend(frameon=False, loc='upper left', fontsize=24); sns.despine(ax=ax_hist, left=True)

    # F. Loss Graph (FIXED KEY HERE)
    ax_loss = fig.add_subplot(gs[2, 0])
    loss_hist = np.array(history.get('tr_loss', history.get('train_loss', [])))
    plot_gradient_line(ax_loss, np.arange(1, len(loss_hist) + 1), loss_hist, 'Train Loss', rising_is_good=False, is_loss=True)

    # G. PR Curves
    ax_pr = fig.add_subplot(gs[2, 2])
    for i in range(n_classes):
        if np.sum(y_bin[:, i]) > 0:
            p, r, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
            ax_pr.plot(r, p, lw=4, color=CLS_COLORS[i % 4], label=f'{classes[i]} ({average_precision_score(y_bin[:, i], y_prob[:, i]):.2f})')
    ax_pr.set_title('Precision-Recall Curves', fontsize=36, fontweight='bold'); ax_pr.legend(loc='lower left', frameon=False, fontsize=25)

    logger.save_fig(fig, "full_report.png"); plt.close(fig)
    return model, loader, classes, history, 0

# SUBMISSION

In [9]:
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, r, tf): self.fs = list(Path(r).glob("*.*")); self.tf = tf
    def __len__(self): return len(self.fs)
    def __getitem__(self, i): return self.tf(Image.open(self.fs[i]).convert('RGB')), self.fs[i].name

def create_submission(models_list, classes, logger, CFG):
    tf = transforms.Compose([
        transforms.Resize((CFG['img_size'], CFG['img_size'])), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    dl = DataLoader(TestDataset(CFG['test_dir'], tf), batch_size=CFG['batch_size'], shuffle=False)

    # TTA Wrappers
    tta_ms = [tta.ClassificationTTAWrapper(m, tta.aliases.flip_transform()) for m in models_list]
    [m.eval() for m in tta_ms]; res = []

    print("Generating Ensemble TTA Submission...")
    with torch.no_grad():
        for x, names in tqdm(dl):
            x = x.to(CFG['device']); probs = torch.zeros(x.size(0), len(classes)).to(CFG['device'])
            for m in tta_ms: probs += torch.softmax(m(x), dim=1)
            [res.append({'filename':f, 'class':classes[p]}) for f,p in zip(names, torch.argmax(probs,1).cpu().numpy())]

    logger.save_csv(pd.DataFrame(res), "submission.csv")

# MAIN

## CFG + PREREQ

In [10]:
def free(): gc.collect(); torch.cuda.empty_cache()

CFG_BASE = {
    'repo_url': 'github.com/kvdep/DL-TESTWORK.git',
    'token': token,
    'data_dir': f'{path_filtered}/dataset_filtered',
    'test_dir': f'{path_sub}/test/test/images',
    'epochs': 5,
    'lr': 1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

In [11]:
clean_duplicates(CFG_BASE['data_dir'], 5)

2025-12-19 15:57:10,095: INFO Start: Calculating hashes...


Deduping datasets\datasets\kvdep1\dumplings\versions\5/dataset_filtered...


100%|██████████| 1662/1662 [00:07<00:00, 218.44it/s]
2025-12-19 15:57:19,068: INFO End: Calculating hashes!
d:\Programs\Python\Lib\site-packages\imagededup\methods\hashing.py:317: RuntimeWarning: Parameter num_enc_workers has no effect since encodings are already provided
  warnings.warn('Parameter num_enc_workers has no effect since encodings are already provided', RuntimeWarning)
2025-12-19 15:57:19,069: INFO Start: Evaluating hamming distances for getting duplicates
2025-12-19 15:57:19,069: INFO Start: Retrieving duplicates using BKTree algorithm
100%|██████████| 1662/1662 [00:05<00:00, 324.06it/s]
2025-12-19 15:57:25,436: INFO End: Retrieving duplicates using BKTree algorithm
2025-12-19 15:57:25,436: INFO End: Evaluating hamming distances for getting duplicates


Removing 132...


In [16]:
logger = GitLogger(CFG_BASE['repo_url'], CFG_BASE['token']); logger.save_code()

[INFO] Code saved to d:\ВУЗ\DL\КР 2\DL-TESTWORK\dumplings\2025-12-19_15-59-08\run_code.py


## EfficientNet

In [17]:
CFG_EFF = CFG_BASE.copy()
CFG_EFF.update({'arch': 'effnet', 'batch_size': 32, 'img_size': 338, 'lr': 1e-3})

In [ ]:
free(); m1, val_dl, cls, h1, b1 = train(CFG_EFF)

Building effnet...


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to weights\checkpoints\efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:04<00:00, 18.5MB/s]


Training effnet...


Ep 1:   0%|          | 0/39 [00:07<?, ?it/s]

In [ ]:
run_full_evaluation(m1, val_dl, logger, h1, cls, CFG_EFF); logger.push("EffNet") ; m1.cpu()

## ConvNeXt

In [ ]:
CFG_NXT = CFG_BASE.copy()
CFG_NXT.update({'arch': 'convnext', 'batch_size': 24, 'img_size': 338, 'lr': 8e-4})


In [ ]:
free(); m2, _, _, h2, b2 = train(CFG_NXT);

In [ ]:
run_full_evaluation(m2, val_dl, logger, h2, cls, CFG_NXT); logger.push("ConvNeXt"); m2.cpu()

## SWIN TRANSFORMER V2

In [ ]:
CFG_SWIN = CFG_BASE.copy()
CFG_SWIN.update({'arch': 'swin', 'batch_size': 32, 'img_size': 256, 'lr': 5e-4})

free()
m3, _, _, h3, b3 = train(CFG_SWIN)
run_full_evaluation(m3, val_dl, logger, h3, cls, CFG_SWIN)
logger.push("SwinV2")

## MaxVIT

In [ ]:
# --- MAXVIT TINY ---
# Стандартный input для MaxViT - 224
CFG_MAX = CFG_BASE.copy()
CFG_MAX.update({'arch': 'maxvit', 'batch_size': 24, 'img_size': 224, 'lr': 5e-4})

free()
m4, _, _, h4, b4 = train(CFG_MAX)
run_full_evaluation(m4, val_dl, logger, h4, cls, CFG_MAX)
logger.push("MaxViT")

m4.cpu() # Освобождаем GPU

## EVA-02

In [ ]:
# --- EVA-02 MIM ---
# Требует 336px или 448px. Модель тяжелая, снижаем batch_size
CFG_EVA = CFG_BASE.copy()
CFG_EVA.update({'arch': 'eva02', 'batch_size': 16, 'img_size': 336, 'lr': 3e-4})

free()
m5, _, _, h5, b5 = train(CFG_EVA)
run_full_evaluation(m5, val_dl, logger, h5, cls, CFG_EVA)
logger.push("EVA02")

m5.cpu() # Освобождаем GPU

## ENSEMBLE

In [ ]:
models_list = [m1, m2, m3, m4, m5]
acc_scores = [b1, b2, b3, b4, b5]
print(f"Ensembling {len(models_list)} models with best val acc: {acc_scores}")
logger.start(max(acc_scores))
free();gpu_models = [m.to(CFG_BASE['device']) for m in models_list]
viz_ensemble(gpu_models, val_dl, cls, logger, CFG_BASE['device'])
create_submission(gpu_models, cls, logger, CFG_EFF)
logger.push("Final_Ensemble_5_Experts")
for m in gpu_models: m.cpu()
free()